In [ ]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import glob
import os
from tensorflow.keras.metrics import RootMeanSquaredError
import numpy as np
from sklearn.preprocessing import MinMaxScaler




2026-02-12 12:25:47.932685: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-12 12:25:47.941237: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 12:25:48.312000: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-12 12:25:49.867684: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [5]:
from keras.models import Model
from keras.layers import Input, Dense, GRU as GRU_layer
from keras.layers import Flatten,  Dropout


#The proposed GRU-FCN model code

def generate_GRU(n_timesteps, n_features, n_outputs):
    """
    Cria um modelo GRU puro.
    
    n_timesteps: Comprimento da sequência temporal (no seu caso, 1)
    n_features: Quantidade de colunas/variáveis de entrada
    n_outputs: Quantidade de valores a prever
    """
    
    # Camada de Entrada
    inp = Input(shape=(n_timesteps, n_features))
    
    # Camada GRU
    # return_sequences=False pois queremos apenas o vetor final para a Dense
    # 64 unidades é um bom ponto de partida (mais robusto que 8)
    x = GRU_layer(64, return_sequences=False)(inp) 
    
    # Dropout para evitar overfitting
    x = Dropout(0.2)(x)
    
    # Camada de Saída
    # activation='linear' é o padrão para regressão. 
    # Se seus dados Y foram normalizados entre 0 e 1, você pode usar 'sigmoid'.
    out = Dense(n_outputs, activation='linear')(x)
    
    model = Model(inputs=inp, outputs=out)
    return model


def generate_MLP_model(max_seq, output):
    ip = Input(shape=(max_seq,))
    y= Dropout(0.1)(ip)
    y = Flatten()(y)
    y = Dense(500, activation='relu')(y)
    y = Dropout(0.2)(y)
    y = Dense(500, activation='relu')(y)
    y = Dropout(0.2)(y)
    y = Dense(500, activation = 'relu')(y)
    y = Dropout(0.3)(y)
    out = Dense(output, activation='linear')(y)
    model = Model(ip, out)
    model.summary()
    return model

In [ ]:
train_hour_wind = pd.read_csv('../../data/Tabelas_criadas/treino_hora.csv')
test_hour_wind = pd.read_csv('../../data/Tabelas_criadas/teste_hora.csv')
val_hour_wind = pd.read_csv('../../data/Tabelas_criadas/val_hora.csv')
inputs_hour = 7*24
outputs_hour = 1*24

train_day_wind = pd.read_csv('../../data/Tabelas_criadas/treino_dia.csv')
test_day_wind = pd.read_csv('../../data/Tabelas_criadas/teste_dia.csv')
val_day_wind = pd.read_csv('../../data/Tabelas_criadas/val_dia.csv')
inputs_day = 7
outputs_day = 1

#Separando os dados por instituição
inst = train_day_wind["id_institution"].unique()
X_train_d = {}
y_train_d = {}
X_test_d = {}
y_test_d = {}
X_val_d = {}
y_val_d = {}

for i in inst:
    train_d = train_day_wind[train_day_wind["id_institution"]==i].reset_index(drop=True)
    train_d = train_d.drop(columns=["id_institution"])
    test_d = test_day_wind[test_day_wind["id_institution"]==i].reset_index(drop=True)
    test_d = test_d.drop(columns=["id_institution"])
    val_d = val_day_wind[val_day_wind["id_institution"]==i].reset_index(drop=True)
    val_d = val_d.drop(columns=["id_institution"])
    np_train_d = np.array(train_d)
    np_test_d = np.array(test_d)
    np_val_d = np.array(val_d)
    X_train_d[i] = np_train_d[:, :inputs_day + 1].astype('float32')
    y_train_d[i] = np_train_d[:, inputs_day+1:].astype('float32')
    X_test_d[i] = np_test_d[:, :inputs_day + 1].astype('float32')
    y_test_d[i] = np_test_d[:, np.r_[inputs_day+1:inputs_day+1+y_train_d[i].shape[1],0]].astype('float32')
    X_val_d[i] = np_val_d[:, :inputs_day + 1].astype('float32')
    y_val_d[i] = np_val_d[:, inputs_day+1:].astype('float32')





